### Подключение PySpark, загрузка библиотек, настройка изображений

In [127]:
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, mean, countDistinct, count, size, when, regexp_extract, split

sns.set_style('darkgrid')
params = {'legend.fontsize': 'medium', 'figure.figsize': (10,8), 'figure.dpi':100, 'axes.labelsize': 'medium', 'axes.titlesize': 'medium', 'xtick.labelsize': 'medium', 'ytick.labelsize': 'medium'}
plt.rcParams.update(params)

### Запуск сессии Pyspark

In [128]:
# spark = SparkSession.builder.appName('EDA films').getOrCreate()

### Загрузка данных и обзор датасета

In [129]:
df_movies = spark.read.csv('sp_movies.csv', header=True, inferSchema=True)
df_ratings = spark.read.csv('sp_ratings.csv', header=True, inferSchema=True)
df_tags = spark.read.csv('sp_tags.csv', header=True, inferSchema=True)

In [130]:
df_movies.show(3)

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
+-------+--------------------+--------------------+
only showing top 3 rows


In [131]:
df_ratings.show(3)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
+------+-------+------+---------+
only showing top 3 rows


In [132]:
df_tags.show(20)

+------+-------+-----------------+----------+
|userId|movieId|              tag| timestamp|
+------+-------+-----------------+----------+
|     2|  60756|            funny|1445714994|
|     2|  60756|  Highly quotable|1445714996|
|     2|  60756|     will ferrell|1445714992|
|     2|  89774|     Boxing story|1445715207|
|     2|  89774|              MMA|1445715200|
|     2|  89774|        Tom Hardy|1445715205|
|     2| 106782|            drugs|1445715054|
|     2| 106782|Leonardo DiCaprio|1445715051|
|     2| 106782|  Martin Scorsese|1445715056|
|     7|  48516|     way too long|1169687325|
|    18|    431|        Al Pacino|1462138765|
|    18|    431|         gangster|1462138749|
|    18|    431|            mafia|1462138755|
|    18|   1221|        Al Pacino|1461699306|
|    18|   1221|            Mafia|1461699303|
|    18|   5995|        holocaust|1455735472|
|    18|   5995|       true story|1455735479|
|    18|  44665|     twist ending|1456948283|
|    18|  52604|  Anthony Hopkins|

### Статистика данных

In [133]:
print('Кол-во пользователей поставивших оценку')
df_ratings.select(countDistinct('userId')).show()

Кол-во пользователей поставивших оценку
+----------------------+
|count(DISTINCT userId)|
+----------------------+
|                   610|
+----------------------+



In [134]:
print('Кол-во оцененных фильмов')
df_ratings.select(countDistinct('movieId')).show()

Кол-во оцененных фильмов
+-----------------------+
|count(DISTINCT movieId)|
+-----------------------+
|                   9724|
+-----------------------+



In [135]:
print('Кол-во фильмов с комментариями (tags)')
df_tags.select(countDistinct('movieId')).show()

Кол-во фильмов с комментариями (tags)
+-----------------------+
|count(DISTINCT movieId)|
+-----------------------+
|                   1572|
+-----------------------+



In [136]:
print('Кол-во комментариев (tags)')
df_tags.select(countDistinct('tag')).show()

Кол-во комментариев (tags)
+-------------------+
|count(DISTINCT tag)|
+-------------------+
|               1589|
+-------------------+



In [137]:
df1 = df_ratings.alias('df1')
df2 = df_tags.alias('df2')
df3 = df_movies.alias('df3')

In [138]:
df3.show()

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
|      6|         Heat (1995)|Action|Crime|Thri...|
|      7|      Sabrina (1995)|      Comedy|Romance|
|      8| Tom and Huck (1995)|  Adventure|Children|
|      9| Sudden Death (1995)|              Action|
|     10|    GoldenEye (1995)|Action|Adventure|...|
|     11|American Presiden...|Comedy|Drama|Romance|
|     12|Dracula: Dead and...|       Comedy|Horror|
|     13|        Balto (1995)|Adventure|Animati...|
|     14|        Nixon (1995)|               Drama|
|     15|Cutthroat Island ...|Action|Adventure|...|
|     16|       Casino (1995)|         Crime|Drama|
|     17|Sen

### Извлечение года из названия

In [139]:
df3 = df3.withColumn('year', regexp_extract(df3['title'], r'\((\d{4})\)', 1))

In [140]:
df3.show()

+-------+--------------------+--------------------+----+
|movieId|               title|              genres|year|
+-------+--------------------+--------------------+----+
|      1|    Toy Story (1995)|Adventure|Animati...|1995|
|      2|      Jumanji (1995)|Adventure|Childre...|1995|
|      3|Grumpier Old Men ...|      Comedy|Romance|1995|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|1995|
|      5|Father of the Bri...|              Comedy|1995|
|      6|         Heat (1995)|Action|Crime|Thri...|1995|
|      7|      Sabrina (1995)|      Comedy|Romance|1995|
|      8| Tom and Huck (1995)|  Adventure|Children|1995|
|      9| Sudden Death (1995)|              Action|1995|
|     10|    GoldenEye (1995)|Action|Adventure|...|1995|
|     11|American Presiden...|Comedy|Drama|Romance|1995|
|     12|Dracula: Dead and...|       Comedy|Horror|1995|
|     13|        Balto (1995)|Adventure|Animati...|1995|
|     14|        Nixon (1995)|               Drama|1995|
|     15|Cutthroat Island ...|A

### Извлечение данных о жанрах

In [147]:
from pyspark.sql import functions as F

# разбиваем строку по "|"
split_expr = F.split(F.col("genres"), "\\|")

# вычисляем реальное максимальное число жанров
max_genres = df3.select(F.max(F.size(split_expr))).first()[0]

# создаём новые колонки genre1..genreN (безопасно через try_element_at)
for i in range(1, max_genres + 1):
    df3 = df3.withColumn(f"genre{i}", F.expr(f"try_element_at(split(genres, '\\\\|'), {i})"))

# список новых колонок
genre_columns = [f"genre{i}" for i in range(1, max_genres + 1)]

# считаем количество непустых жанров
genre_count_expr = sum(
    F.when(F.col(c).isNotNull() & (F.col(c) != ""), 1).otherwise(0)
    for c in genre_columns
)
df3 = df3.withColumn("genre_count", genre_count_expr)

df3.show(truncate=False)

+-------+-------------------------------------+-------------------------------------------+----+-------------------------------------------------+---------+---------+--------+------+--------+------+------+------+------+-------+-----------+
|movieId|title                                |genres                                     |year|genres_arr                                       |genre1   |genre2   |genre3  |genre4|genre5  |genre6|genre7|genre8|genre9|genre10|genre_count|
+-------+-------------------------------------+-------------------------------------------+----+-------------------------------------------------+---------+---------+--------+------+--------+------+------+------+------+-------+-----------+
|1      |Toy Story (1995)                     |Adventure|Animation|Children|Comedy|Fantasy|1995|[Adventure, Animation, Children, Comedy, Fantasy]|Adventure|Animation|Children|Comedy|Fantasy |NULL  |NULL  |NULL  |NULL  |NULL   |5          |
|2      |Jumanji (1995)                 

In [148]:
df3 = df3.drop('genres', 'genres_arr')

In [151]:
df3.show()

+-------+--------------------+----+---------+---------+--------+------+--------+------+------+------+------+-------+-----------+
|movieId|               title|year|   genre1|   genre2|  genre3|genre4|  genre5|genre6|genre7|genre8|genre9|genre10|genre_count|
+-------+--------------------+----+---------+---------+--------+------+--------+------+------+------+------+-------+-----------+
|      1|    Toy Story (1995)|1995|Adventure|Animation|Children|Comedy| Fantasy|  NULL|  NULL|  NULL|  NULL|   NULL|          5|
|      2|      Jumanji (1995)|1995|Adventure| Children| Fantasy|  NULL|    NULL|  NULL|  NULL|  NULL|  NULL|   NULL|          3|
|      3|Grumpier Old Men ...|1995|   Comedy|  Romance|    NULL|  NULL|    NULL|  NULL|  NULL|  NULL|  NULL|   NULL|          2|
|      4|Waiting to Exhale...|1995|   Comedy|    Drama| Romance|  NULL|    NULL|  NULL|  NULL|  NULL|  NULL|   NULL|          3|
|      5|Father of the Bri...|1995|   Comedy|     NULL|    NULL|  NULL|    NULL|  NULL|  NULL|  N

### Анализ данных

In [160]:
rating_avg = df1.groupBy('movieId').agg(mean('rating').alias('rating_avg'))
rating_avg = rating_avg.withColumnRenamed('movieId', 'movieId_avg')
rating_avg.show(3)

+-----------+-----------------+
|movieId_avg|       rating_avg|
+-----------+-----------------+
|       1580|3.487878787878788|
|       2366|             3.64|
|       3175|             3.58|
+-----------+-----------------+
only showing top 3 rows


In [161]:
rating_count = df1.groupBy('movieId').agg(count('rating').alias('rating_count'))
rating_count = rating_count.withColumnRenamed('movieId', 'movieId_count')
rating_count.show(3)

+-------------+------------+
|movieId_count|rating_count|
+-------------+------------+
|         1580|         165|
|         2366|          25|
|         3175|          75|
+-------------+------------+
only showing top 3 rows


In [162]:
user_rating = df1.groupBy('userId').agg(mean('rating').alias('user_rating_avg'))
user_rating = user_rating.withColumnRenamed('userId', 'userId_avg')
user_rating.show(3)

+----------+------------------+
|userId_avg|   user_rating_avg|
+----------+------------------+
|       148|3.7395833333333335|
|       463| 3.787878787878788|
|       471|             3.875|
+----------+------------------+
only showing top 3 rows


In [163]:
user_count = df1.groupBy('userId').agg(count('rating').alias('user_rating_count'))
user_count = user_count.withColumnRenamed('userId', 'userId_count')
user_count.show(3)

+------------+-----------------+
|userId_count|user_rating_count|
+------------+-----------------+
|         148|               48|
|         463|               33|
|         471|               28|
+------------+-----------------+
only showing top 3 rows


In [164]:
df_movie = rating_avg.join(rating_count, col('movieId_avg') == col('movieId_count'), 'inner').drop('movieId_count')
df_movie.show()

+-----------+------------------+------------+
|movieId_avg|        rating_avg|rating_count|
+-----------+------------------+------------+
|       1580| 3.487878787878788|         165|
|       2366|              3.64|          25|
|       3175|              3.58|          75|
|       1088| 3.369047619047619|          42|
|      32460|              4.25|           4|
|      44022| 3.217391304347826|          23|
|      96488|              4.25|           4|
|       1238| 4.055555555555555|           9|
|       1342|               2.5|          11|
|       1591|2.6346153846153846|          26|
|       1645| 3.411764705882353|          51|
|       4519|3.3333333333333335|           9|
|       2142|               2.7|          10|
|        471|              3.55|          40|
|       3997|1.8333333333333333|          12|
|        833|               2.0|           6|
|       3918|3.2777777777777777|           9|
|       7982|              3.25|           4|
|       1959|3.6666666666666665|  

In [165]:
df_user = user_rating.join(user_count, col('userId_avg') == col('userId_count'), 'inner').drop('userId_count')
df_user.show()

+----------+------------------+-----------------+
|userId_avg|   user_rating_avg|user_rating_count|
+----------+------------------+-----------------+
|       148|3.7395833333333335|               48|
|       463| 3.787878787878788|               33|
|       471|             3.875|               28|
|       496| 3.413793103448276|               29|
|       243| 4.138888888888889|               36|
|       392|               3.2|               25|
|       540|               4.0|               42|
|        31|              3.92|               50|
|       516|3.6923076923076925|               26|
|        85|3.7058823529411766|               34|
|       137| 3.978723404255319|              141|
|       251| 4.869565217391305|               23|
|       451|3.7941176470588234|               34|
|       580| 3.529816513761468|              436|
|        65| 4.029411764705882|               34|
|       458|4.1525423728813555|               59|
|        53|               5.0|               20|


In [166]:
df_user.sort(col('user_rating_count').desc()).show()

+----------+------------------+-----------------+
|userId_avg|   user_rating_avg|user_rating_count|
+----------+------------------+-----------------+
|       414| 3.391957005189029|             2698|
|       599|2.6420500403551253|             2478|
|       474| 3.398956356736243|             2108|
|       448|2.8473712446351933|             1864|
|       274| 3.235884101040119|             1346|
|       610|3.6885560675883258|             1302|
|        68| 3.233730158730159|             1260|
|       380|3.6732348111658455|             1218|
|       606|3.6573991031390136|             1115|
|       288|3.1459715639810426|             1055|
|       249|3.6964627151051626|             1046|
|       387|3.2585199610516065|             1027|
|       182|3.5112589559877176|              977|
|       307|2.6656410256410257|              975|
|       603|3.5079533404029695|              943|
|       298| 2.363684771033014|              939|
|       177| 3.375553097345133|              904|


In [168]:
df_movie.sort(col('rating_count').desc()).show()

+-----------+------------------+------------+
|movieId_avg|        rating_avg|rating_count|
+-----------+------------------+------------+
|        356| 4.164133738601824|         329|
|        318| 4.429022082018927|         317|
|        296| 4.197068403908795|         307|
|        593| 4.161290322580645|         279|
|       2571| 4.192446043165468|         278|
|        260| 4.231075697211155|         251|
|        480|              3.75|         238|
|        110| 4.031645569620253|         237|
|        589| 3.970982142857143|         224|
|        527|             4.225|         220|
|       2959| 4.272935779816514|         218|
|          1|3.9209302325581397|         215|
|       1196|4.2156398104265405|         211|
|         50| 4.237745098039215|         204|
|       2858| 4.056372549019608|         204|
|         47|3.9753694581280787|         203|
|        780|3.4455445544554455|         202|
|        150| 3.845771144278607|         201|
|       1198|            4.2075|  

In [169]:
df_movie = df_movie.withColumnRenamed('movieId_avg', 'movieId')

In [170]:
dfk = df3.select('movieId', df3['genre1'])
df2 = df2.join(dfk, on='movieId', how='inner')

In [171]:
df2.show()

+-------+------+----------------+----------+---------+
|movieId|userId|             tag| timestamp|   genre1|
+-------+------+----------------+----------+---------+
|      1|   567|             fun|1525286013|Adventure|
|      1|   474|           pixar|1137206825|Adventure|
|      1|   336|           pixar|1139045764|Adventure|
|      2|   474|            game|1137375552|Adventure|
|      2|    62|  Robin Williams|1528843907|Adventure|
|      2|    62|magic board game|1528843932|Adventure|
|      2|    62|         fantasy|1528843929|Adventure|
|      3|   289|             old|1143424860|   Comedy|
|      3|   289|           moldy|1143424860|   Comedy|
|      5|   474|          remake|1137373903|   Comedy|
|      5|   474|       pregnancy|1137373903|   Comedy|
|      7|   474|          remake|1137375642|   Comedy|
|     11|   474|       president|1137374904|   Comedy|
|     11|   474|        politics|1137374904|   Comedy|
|     14|   474|       president|1137375623|    Drama|
|     14| 

In [172]:
user_tags = df2.groupBy('userId', 'movieId').agg(count('tag').alias('tag_gount'))
user_tags.show()

+------+-------+---------+
|userId|movieId|tag_gount|
+------+-------+---------+
|   474|    412|        1|
|   474|    551|        2|
|   474|   1348|        1|
|   474|   1513|        1|
|   474|   4027|        1|
|   474|   5876|        1|
|   474|   6380|        1|
|   474|   8014|        1|
|   474|  30707|        1|
|    62| 108190|        8|
|   474|     32|        1|
|   474|     43|        1|
|   474|   1797|        1|
|   424|   3499|        7|
|   474|   4117|        1|
|   474|   5644|        2|
|   125|  60950|        1|
|   477|  62336|        4|
|   474|    671|        1|
|   474|   1178|        2|
+------+-------+---------+
only showing top 20 rows
